In [11]:
import pandas as pd
import ollama
import re

np.float64(0.23)

In [5]:
prova_2015 = pd.read_csv(rf"C:\Users\Caio Neves\Desktop\llm-benchmark-tjdft\database_csv\provas_tjdft_com_gabarito.csv")

prova_2015.head()

,Nquestao,enunciado,A,B,C,D,E,questao_completa,id_prova,gabarito
0,1,Considerando as disposições da Lei de Introduç...,Uma lei nova que estabeleça disposições gerais...,"No ordenamento jurídico brasileiro, admite-se ...","Entre as fontes de interpretação das normas, c...",A utilização dos costumes como método de integ...,A lei do país de origem do falecido estrangeir...,Considerando as disposições da Lei de Introduç...,2014,E
1,2,"A respeito dos direitos da personalidade, assi...",O divórcio restitui a situação de incapacidade...,Não se admite declaração de morte presumida se...,A comoriência pode ser reconhecida ainda que o...,"Admite-se, no ordenamento jurídico brasileiro,...","Os ausentes, desde que assim declarados por se...","A respeito dos direitos da personalidade, assi...",2014,C
2,3,"A respeito do direito das obrigações, assinale...",No caso de obrigações divisíveis com pluralida...,A validade da cessão de crédito depende da reg...,As causas suspensivas da prescrição de caráter...,A obrigação solidária não se transmite aos her...,"Em se tratando de obrigação alternativa, silen...","A respeito do direito das obrigações, assinale...",2014,X
3,4,Assinale a opção correta acerca do direito de ...,A principal característica do contrato de segu...,Considerar-se-á aceita a doação com encargo se...,Será nulo o contrato de compra e venda celebra...,"No contrato do tipo emptio spei, o risco assum...",Um contrato de execução instantânea pode ser o...,Assinale a opção correta acerca do direito de ...,2014,X
4,5,Assinale a opção correta a respeito da respons...,O estado de necessidade exclui o dever de inde...,"Consoante entendimento do STJ, cabe indenizaçã...","De acordo com entendimento do STJ, o termo ini...","Em caso de transporte de cortesia, a responsab...",Para o reconhecimento de dano moral decorrente...,Assinale a opção correta a respeito da respons...,2014,D


In [14]:
SYSTEM = (
    """Você é um especialista em Direito brasileiro resolvendo questões de 
    múltipla escolha de provas. Considere a legislação e a 
    jurisprudência brasileiras. Apenas uma alternativa está correta."""
)

TEMPLATE = """Responda a questão abaixo.

Enunciado: {enunciado}
Alternativas: {alternativas}

Responda apenas com a letra da alternativa correta, exatamente neste formato:
Resposta: <letra>"""



In [22]:
def montar_prompt(row, letras="ABCDE"):
    alts = "\n".join(f"{l}) {row[l]}" for l in letras if pd.notna(row.get(l)))
    return TEMPLATE.format(enunciado=row["enunciado"], alternativas=alts)

In [28]:
def extrair_letra(texto):
    achados = re.findall(r"Resposta:\s*\**\s*([A-E])", texto, flags=re.I)
    return achados[-1].upper() if achados else None

In [39]:
def usou_gpu(modelo):
    for m in ollama.ps().models:
        if m.model == modelo:
            return m.size_vram > 0
    return None

In [30]:
x = extrair_letra("Resposta: ** A")
x

'A'

In [ ]:
def perguntar(modelo, prompt):
    r = ollama.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0, "seed": 42, "num_ctx": 4096},
        keep_alive="30m",
    )
    texto = r["message"]["content"]
    pct_gpu, tamanho_gb = usou_gpu(modelo)  
    return {
        "resposta_bruta": texto,
        "letra": extrair_letra(texto),
        "tempo_s": r["total_duration"] / 1e9,
        "tokens_prompt": r["prompt_eval_count"],
        "tokens_saida": r["eval_count"],
        "usou_gpu": usou_gpu(modelo),
    }

In [42]:
# escolha o modelo
modelo = "gemma4:e4b-it-q4_K_M"
resultados = []

for _, row in prova_2015.iterrows():
        out = perguntar(modelo, montar_prompt(row))
        resultados.append({"modelo": modelo, "Nquestao": row["Nquestao"], **out})

  
    
df_res = pd.DataFrame(resultados)

In [45]:
df_final = prova_2015.merge(df_res, on="Nquestao", how="left")
df_final["absoluto_correto"] = df_final["letra"] == df_final["gabarito"]
df_final["absoluto_correto"].sum() / len(df_final) * 100

np.float64(30.097087378640776)

In [51]:
df_final.to_csv("resultados_prova_2015.csv", index=False)